<div align="center">

## Zadanie 1

</div>

Importy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from collections import Counter
from scipy.stats import pearsonr


In [ ]:
def generate_credit_data(n_samples=1000, n_features=10, random_state=42):
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=6,
        n_redundant=2,
        n_clusters_per_class=2,
        random_state=random_state
    )
    return X, y


In [ ]:
def bootstrap_sample(X, y, random_state=None):
    rng = np.random.RandomState(random_state)
    n = len(X)

    indices = rng.choice(n, size=n, replace=True)
    oob_indices = list(set(range(n)) - set(indices))

    return X[indices], y[indices], oob_indices


In [ ]:
def train_single_trees(X_train, y_train, n_trees=10):
    trees = []
    for seed in range(n_trees):
        tree = DecisionTreeClassifier(random_state=seed)
        tree.fit(X_train, y_train)
        trees.append(tree)
    return trees


In [ ]:
def calculate_prediction_variance(trees, X):
    probs = np.array([tree.predict_proba(X)[:, 1] for tree in trees])
    return np.var(probs, axis=0)


In [ ]:
def ensemble_predict(trees, X):
    predictions = np.array([tree.predict(X) for tree in trees])

    final_preds = []
    for i in range(predictions.shape[1]):
        final_preds.append(Counter(predictions[:, i]).most_common(1)[0][0])

    return np.array(final_preds)


In [ ]:
def build_randomized_tree(X_train, y_train, seed=None):
    tree = DecisionTreeClassifier(
        random_state=seed,
        max_features="sqrt"
    )
    tree.fit(X_train, y_train)
    return tree


def train_random_forest(X_train, y_train, n_trees=50):
    trees = []
    for i in range(n_trees):
        tree = build_randomized_tree(X_train, y_train, seed=i)
        trees.append(tree)
    return trees


In [ ]:
def prediction_correlation(trees, X):
    probs = np.array([tree.predict_proba(X)[:, 1] for tree in trees])

    corrs = []
    for i in range(len(trees) - 1):
        c, _ = pearsonr(probs[i], probs[i + 1])
        corrs.append(c)

    return np.mean(corrs)


In [ ]:
def oob_error(X, y, trees, oob_indices_list):
    n = len(X)
    votes = [[] for _ in range(n)]

    for tree, oob_idx in zip(trees, oob_indices_list):
        preds = tree.predict(X[oob_idx])

        for idx, pred in zip(oob_idx, preds):
            votes[idx].append(pred)

    final = np.array([
        Counter(v).most_common(1)[0][0] if len(v) > 0 else 0
        for v in votes
    ])

    mask = np.array([len(v) > 0 for v in votes])
    return np.mean(final[mask] != y[mask])


In [ ]:
def plot_decision_boundary(tree, X, y, title):
    X2 = X[:, :2]   # tylko do rysowania

    x_min, x_max = X2[:, 0].min() - 1, X2[:, 0].max() + 1
    y_min, y_max = X2[:, 1].min() - 1, X2[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200)
    )

    # ✅ TU JEST KLUCZOWA POPRAWKA:
    grid = np.zeros((xx.ravel().shape[0], X.shape[1]))
    grid[:, 0] = xx.ravel()
    grid[:, 1] = yy.ravel()

    ZZ = tree.predict(grid)
    ZZ = ZZ.reshape(xx.shape)

    plt.contourf(xx, yy, ZZ, alpha=0.3)
    plt.scatter(X2[:, 0], X2[:, 1], c=y, s=20)
    plt.title(title)
    plt.show()


In [ ]:
if __name__ == "__main__":

    X, y = generate_credit_data()
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # A
    print("=== (A) Niestabilność pojedynczych drzew ===")
    trees = train_single_trees(X_train, y_train, n_trees=10)    # Trenuje 10 drzew decyzyjnych

    variance = calculate_prediction_variance(trees, X_test)     # Liczmy wariancję każdego punktu testowego
    print("Średnia wariancja:", np.mean(variance))              # Średnia wariancja, na całym zbiorze danych czyli dla każdego punktu i dla każdego drzewa

    plt.hist(variance, bins=50)                                 # Wykres warinacji dla tylko 2 cech. Czemu on jest taki dziwny? Pewnie dlatego, że ma więcej cech niż tylko 2. Taki jest mój take
    plt.title("Rozkład wariancji predykcji")
    plt.show()

    # Grranicze decyzjne dla 5 drzew
    for i in range(5):
        plot_decision_boundary(trees[i], X_test, y_test, f"Granica - Drzewo {i+1}")
        # Wykres warinacji dla tylko 2 cech. Czemu on jest taki dziwny? Pewnie dlatego, że ma więcej cech niż tylko 2. Taki jest mój take

    # B
    print("\n=== (B) Bagging ===")
    boot_trees = []
    for seed in range(1, 51):
        Xb, yb, _ = bootstrap_sample(X_train, y_train, seed)
        tree = DecisionTreeClassifier(random_state=seed)
        tree.fit(Xb, yb)
        boot_trees.append(tree)

    boot_variance = calculate_prediction_variance(boot_trees, X_test)  

    plt.hist(boot_variance, bins=50)
    plt.title("Rozkład wariancji predykcji")
    plt.show()
    print("Średnia wariancja (bez baggingu):", np.mean(variance))
    print("Średnia wariancja (bagging):", np.mean(boot_variance))

    bagging_pred = ensemble_predict(boot_trees, X_test)
    print("Accuracy bagging:", accuracy_score(y_test, bagging_pred))

    # C
    print("\n=== (C) Random Forest - losowe cechy ===")
    rf_trees = train_random_forest(X_train, y_train, n_trees=50)

    corr_plain = prediction_correlation(trees, X_test)
    corr_rf = prediction_correlation(rf_trees, X_test)

    print("Korelacja drzew bez losowania cech:", corr_plain)
    print("Korelacja drzew z losowaniem cech:", corr_rf)

    print("\n=== (D) Out-Of-Bag Error ===")
    oob_trees = []
    oob_indices_list = []

    for i in range(100):
        Xb, yb, oob_idx = bootstrap_sample(X_train, y_train, random_state=i)
        tree = DecisionTreeClassifier()
        tree.fit(Xb, yb)

        oob_trees.append(tree)
        oob_indices_list.append(oob_idx)

    oob = oob_error(X_train, y_train, oob_trees, oob_indices_list)
    print("OOB error:", oob)

    print("\n=== Nasycenie liczby drzew ===")
    for n in [10, 50, 100, 200, 500]:
        rf = RandomForestClassifier(n_estimators=n, max_features="sqrt")
        rf.fit(X_train, y_train)
        acc = rf.score(X_test, y_test)
        print("Drzewa:", n, "Accuracy:", acc)

    print("\n=== Porównanie z RandomForestClassifier ===")
    rf_sklearn = RandomForestClassifier(n_estimators=100, max_features="sqrt")
    rf_sklearn.fit(X_train, y_train)

    print("Accuracy sklearn RF:", rf_sklearn.score(X_test, y_test))


<div align="center">

## Zadanie 2

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import digamma

from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score


In [ ]:
def generate_network_data(n_normal=950, n_anomaly=50, random_state=42):
    np.random.seed(random_state)

    normal = np.vstack([
        np.random.randn(n_normal//3, 4) * 0.5 + [50, 100, 0.8, 30],
        np.random.randn(n_normal//3, 4) * 0.5 + [55, 110, 0.75, 35],
        np.random.randn(n_normal//3, 4) * 0.5 + [48, 95, 0.85, 28]
    ])

    anomalies = np.vstack([
        np.random.randn(n_anomaly//2, 4) * 0.3 + [200, 500, 0.1, 5],    # DDoS
        np.random.randn(n_anomaly//2, 4) * 0.3 + [10, 10, 0.99, 100],   # Port scan
    ])

    X = np.vstack([normal, anomalies])
    y = np.array([0]*len(normal) + [1]*len(anomalies))

    return X, y


In [ ]:
class IsolationTree:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, current_depth=0):
        n_samples, n_features = X.shape

        if n_samples <= 1 or (self.max_depth is not None and current_depth >= self.max_depth):
            return {
                "type": "leaf",
                "size": n_samples,
                "depth": current_depth
            }

        feature = np.random.randint(0, n_features)
        min_val = X[:, feature].min()
        max_val = X[:, feature].max()

        if min_val == max_val:
            return {
                "type": "leaf",
                "size": n_samples,
                "depth": current_depth
            }

        threshold = np.random.uniform(min_val, max_val)

        left = X[X[:, feature] < threshold]
        right = X[X[:, feature] >= threshold]

        return {
            "type": "node",
            "feature": feature,
            "threshold": threshold,
            "left": self.fit(left, current_depth + 1),
            "right": self.fit(right, current_depth + 1)
        }

    def path_length(self, x, node=None, current_depth=0):
        if node is None:
            node = self.root

        if node["type"] == "leaf":
            return current_depth

        if x[node["feature"]] < node["threshold"]:
            return self.path_length(x, node["left"], current_depth + 1)
        else:
            return self.path_length(x, node["right"], current_depth + 1)


In [ ]:
def c_factor(n):
    if n <= 1:
        return 0
    return 2 * (np.log(n - 1) + 0.5772156649) - 2 * (n - 1) / n


In [ ]:
def anomaly_score(path_lengths, n_samples):
    c = c_factor(n_samples)
    return 2 ** (-np.array(path_lengths) / c)


In [ ]:
def build_isolation_forest(X, n_trees=100, max_samples=256):
    trees = []
    for _ in range(n_trees):
        idx = np.random.choice(len(X), size=max_samples, replace=False)
        X_sub = X[idx]

        tree = IsolationTree(max_depth=int(np.ceil(np.log2(max_samples))))
        tree.root = tree.fit(X_sub)
        trees.append(tree)

    return trees


In [ ]:
def average_path_length(trees, X):
    avg_lengths = []

    for x in X:
        lengths = [tree.path_length(x) for tree in trees]
        avg_lengths.append(np.mean(lengths))

    return np.array(avg_lengths)


In [ ]:
X, y = generate_network_data()

trees_single = build_isolation_forest(X, n_trees=1)
paths = [trees_single[0].path_length(x) for x in X]

plt.hist(np.array(paths)[y == 0], bins=30, alpha=0.6, label="Normal")
plt.hist(np.array(paths)[y == 1], bins=30, alpha=0.6, label="Anomaly")
plt.legend()
plt.title("Histogram głębokości izolacji")
plt.show()


In [ ]:
np.random.seed(0)
normal_2d = np.random.randn(200, 2)
anomalies_2d = np.array([[5,5],[6,6],[7,7]])
X2D = np.vstack([normal_2d, anomalies_2d])
y2D = np.array([0]*200 + [1]*3)

tree2D = IsolationTree(max_depth=10)
tree2D.root = tree2D.fit(X2D)

plt.scatter(X2D[:-3, 0], X2D[:-3, 1])
plt.scatter(anomalies_2d[:, 0], anomalies_2d[:, 1], color="red")
plt.title("Anomalie izolowane po kilku podziałach")
plt.show()

depths_2d = np.array([tree2D.path_length(x) for x in X2D])

print("Normal 2D - średnia głębokość:", round(depths_2d[y2D == 0].mean(), 2))
print("Anomalie 2D - średnia głębokość:", round(depths_2d[y2D == 1].mean(), 2))

plt.hist(depths_2d[:200], bins=30, alpha=0.6, label="Normalne")
plt.hist(depths_2d[200:], bins=30, alpha=0.6, label="Anomalie")
plt.legend()
plt.title("Głębokość izolacji: normalne vs anomalie")
plt.show()



In [ ]:
trees = build_isolation_forest(X, n_trees=100, max_samples=256)
paths = average_path_length(trees, X)
scores = anomaly_score(paths, n_samples=256)

threshold = np.percentile(scores, 95)
preds = (scores >= threshold).astype(int)


In [ ]:
print("\n==== METRYKI ====")
print("Precision:", precision_score(y, preds))
print("Recall:", recall_score(y, preds))
print("F1:", f1_score(y, preds))



In [ ]:
lof = LocalOutlierFactor(n_neighbors= 5, contamination=0.05)
lof_preds = (lof.fit_predict(X) == -1).astype(int)

print("\nLOF:")
print("Precision:", precision_score(y, lof_preds))
print("Recall:", recall_score(y, lof_preds))
print("F1:", f1_score(y, lof_preds))


In [ ]:
ocsvm = OneClassSVM(nu=0.05)
svm_preds = (ocsvm.fit_predict(X) == -1).astype(int)

print("\nOne-Class SVM:")
print("Precision:", precision_score(y, svm_preds))
print("Recall:", recall_score(y, svm_preds))
print("F1:", f1_score(y, svm_preds))


In [ ]:
sk_if = IsolationForest(contamination=0.05, random_state=0)
sk_preds = (sk_if.fit_predict(X) == -1).astype(int)

print("\nSklearn IsolationForest:")
print("Precision:", precision_score(y, sk_preds))
print("Recall:", recall_score(y, sk_preds))
print("F1:", f1_score(y, sk_preds))


In [ ]:
for n_trees in [50, 100, 200]:
    for max_samples in [64, 128, 256]:
        trees = build_isolation_forest(X, n_trees=n_trees, max_samples=max_samples)
        paths = average_path_length(trees, X)
        scores = anomaly_score(paths, max_samples)
        preds = (scores >= np.percentile(scores, 95)).astype(int)

        f1 = f1_score(y, preds)
        print(f"Drzewa={n_trees}, próbki={max_samples}, F1={f1:.3f}")


<div align="center">

## Zadanie 3

</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, make_circles
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC


In [ ]:
def make_spirals(n_samples=500, noise=0.1, random_state=42):
    np.random.seed(random_state)
    n = n_samples // 2
    theta = np.linspace(0, 4*np.pi, n)
    r = theta / (4*np.pi)

    x1 = r * np.cos(theta) + noise * np.random.randn(n)
    y1 = r * np.sin(theta) + noise * np.random.randn(n)

    x2 = -r * np.cos(theta) + noise * np.random.randn(n)
    y2 = -r * np.sin(theta) + noise * np.random.randn(n)

    X = np.vstack([
        np.column_stack([x1, y1]),
        np.column_stack([x2, y2])
    ])

    y = np.array([0]*n + [1]*n)
    return X, y


In [ ]:
def plot_decision_boundary(model, X, y, ax, title=""):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h)
    )

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.4, cmap="RdYlBu")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="RdYlBu", edgecolors="black")
    ax.set_title(title)


In [ ]:
def visualize_depths(X, y, dataset_name):
    depths = [1, 3, 5, 10, None]

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    for ax, depth in zip(axes, depths):
        tree = DecisionTreeClassifier(max_depth=depth)
        tree.fit(X, y)

        title = f"depth={depth}" if depth else "depth=None"
        plot_decision_boundary(tree, X, y, ax, title)

    fig.suptitle(f"Granice decyzyjne - {dataset_name}")
    plt.show()


In [ ]:
def visualize_tree_splits(tree, X, y, ax):
    plot_decision_boundary(tree, X, y, ax, "Podziały drzewa")

    t = tree.tree_
    stack = [(0, bounds(X))]

    while stack:
        node_id, ((xmin, xmax), (ymin, ymax)) = stack.pop()

        if t.feature[node_id] == -2:
            continue

        feature = t.feature[node_id]
        threshold = t.threshold[node_id]

        if feature == 0:
            ax.plot([threshold, threshold], [ymin, ymax], "--k")
            left_bounds = ((xmin, threshold), (ymin, ymax))
            right_bounds = ((threshold, xmax), (ymin, ymax))

        elif feature == 1:
            ax.plot([xmin, xmax], [threshold, threshold], "--k")
            left_bounds = ((xmin, xmax), (ymin, threshold))
            right_bounds = ((xmin, xmax), (threshold, ymax))

        stack.append((t.children_left[node_id], left_bounds))
        stack.append((t.children_right[node_id], right_bounds))


def bounds(X):
    return ((X[:, 0].min(), X[:, 0].max()),
            (X[:, 1].min(), X[:, 1].max()))


In [ ]:
def count_splits_for_circle(radius, tolerance):
    true_area = np.pi * radius ** 2
    approx_area = 0
    splits = 0

    while abs(true_area - approx_area) > tolerance:
        splits += 1
        approx_area = (2 * radius) ** 2 / splits

    return splits


In [ ]:
def oblique_split(X, y):
    best_acc = 0
    best_params = None

    for _ in range(200):
        a1, a2 = np.random.uniform(-1, 1, 2)
        t = np.random.uniform(-1, 1)

        preds = (a1 * X[:, 0] + a2 * X[:, 1] < t).astype(int)
        acc = np.mean(preds == y)

        if acc > best_acc:
            best_acc = acc
            best_params = (a1, a2, t)

    return best_params


In [ ]:
if __name__ == "__main__":

    # === DANE ===
    X_moons, y_moons = make_moons(n_samples=500, noise=0.2)
    X_circles, y_circles = make_circles(n_samples=500, noise=0.1, factor=0.5)
    X_spiral, y_spiral = make_spirals()

    # === (a) GRANICE VS GŁĘBOKOŚĆ ===
    visualize_depths(X_moons, y_moons, "Moons")
    visualize_depths(X_circles, y_circles, "Circles")
    visualize_depths(X_spiral, y_spiral, "Spiral")

    # === (b) PODZIAŁY PROSTOPADŁE DO OSI ===
    fig, ax = plt.subplots(figsize=(6, 6))
    tree = DecisionTreeClassifier(max_depth=3)
    tree.fit(X_moons, y_moons)
    visualize_tree_splits(tree, X_moons, y_moons, ax)
    plt.title("Podziały osiowe - głębokość 3")
    plt.show()

    # === PORÓWNANIE Z SVM RBF ===
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    tree = DecisionTreeClassifier(max_depth=5)
    tree.fit(X_moons, y_moons)
    plot_decision_boundary(tree, X_moons, y_moons, axes[0], "Decision Tree")

    svm = SVC(kernel="rbf")
    svm.fit(X_moons, y_moons)
    plot_decision_boundary(svm, X_moons, y_moons, axes[1], "SVM RBF")

    plt.show()

    # === (c) ILE PODZIAŁÓW DLA OKRĘGU ===
    for eps in [5.0, 2.0, 1.0, 0.5]:
        splits = count_splits_for_circle(radius=1.0, tolerance=eps)
        print(f"tolerance={eps}, splits={splits}")

    # === (d) OBLIQUE SPLIT ===
    a1, a2, t = oblique_split(X_moons, y_moons)
    print("Najlepszy ukośny podział:", a1, a2, t)

    xx, yy = np.meshgrid(
        np.linspace(X_moons[:,0].min()-0.5, X_moons[:,0].max()+0.5, 200),
        np.linspace(X_moons[:,1].min()-0.5, X_moons[:,1].max()+0.5, 200)
    )

    Z = (a1 * xx + a2 * yy < t).astype(int)

    plt.contourf(xx, yy, Z, alpha=0.4)
    plt.scatter(X_moons[:,0], X_moons[:,1], c=y_moons, edgecolors="black")
    plt.title("Ukośna granica decyzyjna")
    plt.show()
